# Metro Traffic Forecasting

Predict the next traffic observation from the preceding 24 records with LSTM/GRU.

The data contains repeated timestamps and gaps: **24 records are not necessarily 24 hours**.
This edition selects models using validation only, fits preprocessing on training data,
and keeps equal timestamps in one partition. Historical metrics and plots are in
`RESULTS.md`; they are not outputs of this revised notebook.

Authors of the original experiment: Sergey Kokorev, Nikita Kuznetsov, Georgy Uspensky.

**Run:** install `requirements.txt`, download the Metro Interstate Traffic Volume CSV,
set `DATA_PATH`, then Run All. In Colab upload the CSV with the Files sidebar.
The default run trains one configuration; enable the optional searches for a full comparison.


In [ ]:
from pathlib import Path
import copy
import itertools
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path('Metro_Interstate_Traffic_Volume.csv')
OUTPUT_DIR = Path('artifacts')
SEED = 42
WINDOW = 24
RUN_OPTIMIZER_SWEEP = False
RUN_GRID_SEARCH = False
OUTPUT_DIR.mkdir(exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## Data and chronological split

We preserve the original next-record task, including repeated timestamps.
The forecast horizon is therefore variable and can be zero for consecutive records of
one hour. This is **not a validated one-hour-ahead forecasting benchmark**.
For that task, aggregate duplicate hours and restrict windows to continuous hourly data,
then train and evaluate again. The filter `rain_1h < 40` follows the original experiment;
its threshold is a preprocessing assumption, not a tuned result.


In [ ]:
data = pd.read_csv(DATA_PATH)
required = {'date_time', 'traffic_volume', 'weather_main', 'holiday',
            'temp', 'rain_1h', 'snow_1h', 'clouds_all'}
missing = required - set(data.columns)
if missing:
    raise ValueError(f'Missing columns: {sorted(missing)}')
data['date_time'] = pd.to_datetime(data['date_time'], errors='raise')
data = data.loc[data['rain_1h'] < 40].copy()
data = data.sort_values('date_time', kind='stable').reset_index(drop=True)
print('Rows:', len(data), 'Repeated timestamps:', data['date_time'].duplicated().sum())
print('Timestamp differences:', data['date_time'].diff().value_counts().head())
if not data['date_time'].diff().iloc[1:].eq(pd.Timedelta(hours=1)).all():
    warnings.warn('Irregular observation series: windows and lags are measured in records.')

dt = data['date_time']
for name, values, period in [('hour', dt.dt.hour, 24),
                             ('weekday', dt.dt.weekday, 7),
                             ('month', dt.dt.month - 1, 12)]:
    data[f'sin_{name}'] = np.sin(2 * np.pi * values / period)
    data[f'cos_{name}'] = np.cos(2 * np.pi * values / period)
# Holidays are calendar-known covariates, not inferred from the traffic target.
holiday = data['holiday'].fillna('no').replace(['None', 'no', ''], np.nan)
data['is_holiday'] = holiday.groupby(dt.dt.normalize()).transform(
    lambda s: int(s.notna().any())).astype('int8')

# Split timestamp groups, keeping duplicate hours out of different partitions.
timestamps = data['date_time'].drop_duplicates().to_numpy()
train_cut = timestamps[int(len(timestamps) * 0.70)]
val_cut = timestamps[int(len(timestamps) * 0.85)]
train_df = data.loc[data.date_time < train_cut].copy()
val_df = data.loc[(data.date_time >= train_cut) & (data.date_time < val_cut)].copy()
test_df = data.loc[data.date_time >= val_cut].copy()
assert train_df.date_time.max() < val_df.date_time.min()
assert val_df.date_time.max() < test_df.date_time.min()
assert min(map(len, [train_df, val_df, test_df])) > WINDOW

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_df['weather_main'] = encoder.fit_transform(train_df[['weather_main']]).ravel()
for frame in [val_df, test_df]:
    frame['weather_main'] = encoder.transform(frame[['weather_main']]).ravel()
# Ordinal encoding preserves the original model design; one-hot encoding is an alternative.


In [ ]:
feature_cols = [
    'temp',
    'rain_1h',
    'snow_1h',
    'clouds_all',
    'is_holiday',
    'sin_hour',
    'cos_hour',
    'sin_weekday',
    'cos_weekday',
    'sin_month',
    'cos_month',
    'weather_main',
    'traffic_volume'
]

target = 'traffic_volume'

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

train_X_tab = scaler_X.fit_transform(train_df[feature_cols])
val_X_tab = scaler_X.transform(val_df[feature_cols])
test_X_tab = scaler_X.transform(test_df[feature_cols])

train_y_tab = scaler_y.fit_transform(train_df[[target]])
val_y_tab = scaler_y.transform(val_df[[target]])
test_y_tab = scaler_y.transform(test_df[[target]])

In [ ]:
def make_sequences(X, y, window_size):
    X_seq, y_seq = [], []

    for i in range(len(X) - window_size):
        X_seq.append(X[i:i + window_size])
        y_seq.append(y[i + window_size])

    return np.array(X_seq), np.array(y_seq)

In [ ]:
window_size = WINDOW

X_train, y_train = make_sequences(train_X_tab, train_y_tab, window_size)
X_val, y_val = make_sequences(val_X_tab, val_y_tab, window_size)
X_test, y_test = make_sequences(test_X_tab, test_y_tab, window_size)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TrafficDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
batch_size = 64

train_dataset = TrafficDataset(X_train, y_train)
val_dataset = TrafficDataset(X_val, y_val)
test_dataset = TrafficDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.nn as nn

class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (h_n, c_n) = self.lstm(x)

        # берём выход последнего шага по времени
        last_output = out[:, -1, :]   # shape: (batch, hidden_size)
        y_pred = self.fc(last_output) # shape: (batch, 1)

        return y_pred

In [ ]:
import torch.optim as optim

class GRURegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.0):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, _ = self.gru(x)

        last_output = out[:, -1, :]   # shape: (batch, hidden_size)
        y_pred = self.fc(last_output) # shape: (batch, 1)

        return y_pred

In [ ]:
def train(model, loader, criterion, optimizer, device, l1_lambda=0.0):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        if l1_lambda > 0:
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)

In [ ]:
@torch.no_grad()
def val(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    preds = []
    targets = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_pred = model(X_batch)

        preds.append(y_pred.cpu().numpy())
        targets.append(y_batch.numpy())

    preds = np.vstack(preds)
    targets = np.vstack(targets)

    return preds, targets

## Optional optimizer and regularization comparison

24 configurations: 2 architectures × 4 optimizers × 3 regularization settings.
All models share dropout=0.2; “no penalty” means no added L1/L2 penalty.
Selection uses validation RMSE. Test data is not evaluated inside the sweep.


In [ ]:
def optimizer_sweep():
    architectures = {'LSTM': LSTMRegressor, 'GRU': GRURegressor}
    optimizers = {
        'SGD': lambda params, wd: optim.SGD(params, lr=1e-3, weight_decay=wd),
        'Momentum': lambda params, wd: optim.SGD(params, lr=1e-3, momentum=0.9, weight_decay=wd),
        'RMSProp': lambda params, wd: optim.RMSprop(params, lr=1e-3, weight_decay=wd),
        'Adam': lambda params, wd: optim.Adam(params, lr=1e-3, weight_decay=wd),
    }
    rows = []
    for arch, cls in architectures.items():
        for opt_name, opt_factory in optimizers.items():
            for penalty, l1, l2 in [('None', 0.0, 0.0), ('L1', 1e-4, 0.0), ('L2', 0.0, 1e-4)]:
                seed_everything()
                candidate = cls(X_train.shape[2], 64, 2, 0.2).to(device)
                optimizer = opt_factory(candidate.parameters(), l2)
                for _ in range(15):
                    train(candidate, train_loader, nn.MSELoss(), optimizer, device, l1)
                pred, true = predict(candidate, val_loader, device)
                pred, true = scaler_y.inverse_transform(pred), scaler_y.inverse_transform(true)
                rows.append({'architecture': arch, 'optimizer': opt_name, 'penalty': penalty,
                             'validation_rmse': float(np.sqrt(mean_squared_error(true, pred)))})
    return pd.DataFrame(rows).sort_values('validation_rmse')

if RUN_OPTIMIZER_SWEEP:
    sweep_results = optimizer_sweep()
    sweep_results.to_csv(OUTPUT_DIR / 'optimizer_sweep.csv', index=False)
    display(sweep_results)


## LSTM training and optional grid search

The default configuration comes from the historical experiment. A single run makes
this notebook practical to execute. Optional grid search includes the original 48
configurations (some single-layer dropout choices are functionally equivalent).
Early stopping and checkpoint selection use validation Huber loss only.


In [ ]:
def get_loss_function(loss_name):
    if loss_name == 'mse':
        return nn.MSELoss()
    elif loss_name == 'mae':
        return nn.L1Loss()
    elif loss_name == 'huber':
        return nn.HuberLoss(delta=1.0)
    else:
        raise ValueError(f'Unknown loss: {loss_name}')

In [ ]:
def make_loaders(X_train, y_train, X_val, y_val, batch_size):
    train_dataset = TrafficDataset(X_train, y_train)
    val_dataset = TrafficDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

In [ ]:
def train_model_with_params(
    X_train, y_train,
    X_val, y_val,
    input_size,
    device,
    params,
    max_epochs=20,
    patience=5
):
    train_loader, val_loader = make_loaders(
        X_train, y_train, X_val, y_val, params['batch_size']
    )

    model = LSTMRegressor(
        input_size=input_size,
        hidden_size=params['hidden_size'],
        num_layers=params['num_layers'],
        dropout=params['dropout']
    ).to(device)

    criterion = get_loss_function(params['loss_name'])
    optimizer = optim.Adam(model.parameters(), lr=params['lr'])

    best_val_loss = float('inf')
    best_state = None
    best_epoch = 0
    patience_counter = 0

    history = {
        'train_loss': [],
        'val_loss': []
    }

    for epoch in range(max_epochs):
        train_loss = train(model, train_loader, criterion, optimizer, device)
        val_loss = val(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)

    return {
        'model': model,
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'history': history,
        'params': params
    }

In [ ]:
param_grid = {
    'hidden_size': [64, 96, 128],
    'num_layers': [1, 2],
    'dropout': [0.0, 0.2],
    'lr': [1e-3, 5e-4],
    'batch_size': [64, 128],
    'loss_name': ['huber']
}

In [ ]:
def expand_grid(param_grid):
    keys = list(param_grid.keys())
    values = list(param_grid.values())

    combinations = []
    for combination in itertools.product(*values):
        combinations.append(dict(zip(keys, combination)))

    return combinations

In [ ]:
all_params = expand_grid(param_grid) if RUN_GRID_SEARCH else [{
    'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2,
    'lr': 1e-3, 'batch_size': 64, 'loss_name': 'huber'
}]
best_run, best_score = None, float('inf')
search_rows = []
for i, params in enumerate(all_params, 1):
    seed_everything()
    run = train_model_with_params(X_train, y_train, X_val, y_val,
                                 X_train.shape[2], device, params,
                                 max_epochs=20, patience=5)
    search_rows.append({**params, 'validation_loss': run['best_val_loss'],
                        'best_epoch': run['best_epoch']})
    print(i, '/', len(all_params), search_rows[-1])
    if run['best_val_loss'] < best_score:
        best_run, best_score = run, run['best_val_loss']
pd.DataFrame(search_rows).to_csv(OUTPUT_DIR / 'validation_search.csv', index=False)
torch.save({'state_dict': best_run['model'].state_dict(),
            'params': best_run['params'], 'features': feature_cols,
            'seed': SEED}, OUTPUT_DIR / 'model.pt')
import joblib
joblib.dump({'X': scaler_X, 'y': scaler_y, 'weather_encoder': encoder,
             'feature_cols': feature_cols, 'window': WINDOW,
             'train_cut': train_cut, 'val_cut': val_cut}, OUTPUT_DIR / 'preprocessing.joblib')


## Final test evaluation

Evaluate only after validation choices are frozen. Baselines use the last observation
and the observation 24 records back, on the same targets as the neural network.
Do not change the model in response to these test metrics.


In [ ]:
test_loader = DataLoader(TrafficDataset(X_test, y_test), batch_size=64, shuffle=False)
pred_scaled, true_scaled = predict(best_run['model'], test_loader, device)
y_pred = scaler_y.inverse_transform(pred_scaled).ravel()
y_true = scaler_y.inverse_transform(true_scaled).ravel()
traffic_idx = feature_cols.index('traffic_volume')
# Invert the feature scaler explicitly, rather than assuming X and y scales match.
last = X_test[:, -1, traffic_idx] * scaler_X.scale_[traffic_idx] + scaler_X.mean_[traffic_idx]
lag24 = X_test[:, 0, traffic_idx] * scaler_X.scale_[traffic_idx] + scaler_X.mean_[traffic_idx]
def metrics(true, pred):
    return {'MAE': float(mean_absolute_error(true, pred)),
            'RMSE': float(np.sqrt(mean_squared_error(true, pred))),
            'R2': float(r2_score(true, pred))}
rows = [{'model': name, **metrics(y_true, pred)} for name, pred in
        [('LSTM', y_pred), ('Last observation', last), ('Lag 24 observations', lag24)]]
summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT_DIR / 'test_metrics.csv', index=False)
display(summary)
pd.DataFrame({'timestamp': test_df.date_time.iloc[WINDOW:].to_numpy(),
              'actual': y_true, 'predicted': y_pred, 'last_observation': last,
              'lag24': lag24}).to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y_true[:300], label='Observed')
ax.plot(y_pred[:300], label='LSTM')
ax.set(xlabel='Test observation index', ylabel='Traffic volume',
       title='Next-observation prediction')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'forecast.png', dpi=160)
plt.show()


## Conditional recursive evaluation

Replace traffic lags with predictions while retaining recorded weather covariates.
This measures error accumulation **conditional on observed weather**, not a fully
ex-ante forecast. The selected LSTM is used consistently here.


In [ ]:
@torch.no_grad()
def recursive_predict_test(
    model,
    test_X_tab,
    test_df,
    feature_cols,
    target,
    window_size,
    scaler_X,
    scaler_y,
    device
):
    model.eval()

    traffic_idx = feature_cols.index('traffic_volume')

    # копия тестовых признаков, которую будем постепенно портить своими прогнозами
    X_roll = test_X_tab.copy()

    preds_scaled_y = []

    # идём по времени: предсказываем точку t по окну [t-window_size, ..., t-1]
    for t in range(window_size, len(X_roll)):
        window = X_roll[t - window_size:t]  # shape: (window_size, n_features)
        window_tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)

        pred_scaled_y = model(window_tensor).cpu().numpy()[0, 0]
        preds_scaled_y.append(pred_scaled_y)

        # переводим прогноз из шкалы y в реальные значения
        pred_real = scaler_y.inverse_transform([[pred_scaled_y]])[0, 0]

        # теперь переводим это же значение в шкалу X для столбца traffic_volume
        pred_scaled_x = (pred_real - scaler_X.mean_[traffic_idx]) / scaler_X.scale_[traffic_idx]

        # подменяем traffic_volume в текущей строке t,
        # чтобы на следующих шагах модель использовала уже свой прогноз
        X_roll[t, traffic_idx] = pred_scaled_x

    preds_scaled_y = np.array(preds_scaled_y).reshape(-1, 1)

    # истинные значения цели для сравнения
    y_true_real = test_df[target].iloc[window_size:].values.reshape(-1, 1)

    # предсказания в реальной шкале
    y_pred_real = scaler_y.inverse_transform(preds_scaled_y)

    return y_true_real, y_pred_real

In [ ]:
y_true_recursive, y_pred_recursive = recursive_predict_test(
    model=best_run['model'],
    test_X_tab=test_X_tab,
    test_df=test_df,
    feature_cols=feature_cols,
    target=target,
    window_size=window_size,
    scaler_X=scaler_X,
    scaler_y=scaler_y,
    device=device
)

In [ ]:
recursive_metrics = metrics(y_true_recursive, y_pred_recursive)
print('Conditional recursive evaluation:', recursive_metrics)
import json
(OUTPUT_DIR / 'recursive_metrics.json').write_text(json.dumps(recursive_metrics, indent=2))
